In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
import os
import sys
import camb
import h5py
import healpy as hp
import numpy as np
from astropy import units as u
from ksw import KSW, Cosmology, Data, Shape
import logging

sys.path.append(os.path.join(os.getcwd(), "scripts"))
from utils import Config
from utils.plots import plot_ksw_predictions
from almgen import remove_mono_dipole

logging.getLogger("healpy").setLevel(logging.WARNING)

## KSW-joblib

In [18]:
from joblib import Parallel, delayed

from ksw import utils, estimator_core

logger = logging.getLogger("KSW2")


def process_batch_step(
    tidx_start,
    theta_batch,
    thetas,
    theta_weights,
    lmax,
    dtype,
    rule,
    weights,
    f_i_ell,
    a_ell_m,
    grad_t,
    nphi,
):
    # side affects, needed and read only
    grad_t = grad_t.copy()

    thetas_batch = thetas[tidx_start : tidx_start + theta_batch]
    ct_weights_batch = theta_weights[tidx_start : tidx_start + theta_batch]
    y_m_ell = estimator_core.compute_ylm(thetas_batch, lmax, dtype=dtype)
    estimator_core.step(
        ct_weights_batch,
        rule,
        weights,
        f_i_ell.copy(),
        a_ell_m.copy(),
        y_m_ell,
        grad_t,
        nphi,
    )
    return grad_t


def process_batch_estimate(
    tidx_start,
    theta_batch,
    thetas,
    theta_weights,
    lmax,
    dtype,
    rule,
    weights,
    f_i_ell,
    a_ell_m,
    nphi,
):
    thetas_batch = thetas[tidx_start : tidx_start + theta_batch]
    ct_weights_batch = theta_weights[tidx_start : tidx_start + theta_batch]
    y_m_ell = estimator_core.compute_ylm(thetas_batch, lmax, dtype=dtype)
    return estimator_core.compute_estimate(
        ct_weights_batch, rule, weights, f_i_ell.copy(), a_ell_m.copy(), y_m_ell, nphi
    )


class KSW2(KSW):
    """
    Jupyter doesnt like MPI, so this uses joblib to parallelize the loop
    """

    def _step(self, alm, theta_batch=25):
        alm = utils.alm_return_2d(alm, self.npol, self.lmax)
        alm = self.icov(alm)

        # import matplotlib.pyplot as plt

        # # plt.figure()
        # plt.clf()
        # plt.loglog(alm)
        # plt.xlabel("Multipole moment (l)")
        # plt.ylabel("Power")
        # plt.title("Noise power spectrum")
        # plt.show()

        a_ell_m = utils.alm2a_ell_m(alm)
        a_ell_m = a_ell_m.astype(self.cdtype)
        grad_t = np.zeros_like(a_ell_m)

        red_bisp = self.red_bispectra[0]
        f_i_ell, rule, weights = self._init_reduced_bispectrum(red_bisp)

        # Use joblib to parallelize the loop
        results = Parallel(n_jobs=-1, return_as="generator")(
            delayed(process_batch_step)(
                tidx_start,
                theta_batch,
                self.thetas,
                self.theta_weights,
                self.lmax,
                self.dtype,
                rule,
                weights,
                f_i_ell,
                a_ell_m,
                grad_t,
                self.nphi,
            )
            for tidx_start in range(0, len(self.thetas), theta_batch)
        )
        grad_t = sum(results)

        # Turn back into healpy shape.
        grad_t = utils.a_ell_m2alm(grad_t).astype(self.cdtype)
        return grad_t

    def _process_file_step(self, alm_loader, alm_file, **kwargs):
        logger.info(f"processing {alm_file}")
        alm = alm_loader(alm_file)
        grad_t = self._step(alm, **kwargs)
        mc_gt_sq = utils.contract_almxblm(grad_t, self.icov(self.beam(np.conj(grad_t))))
        return grad_t, mc_gt_sq

    def step_batch(self, alm_loader, alm_files, **kwargs):
        # Monte carlo quantities local to rank.
        mc_idx_loc = 0
        mc_gt_sq_loc = 0
        mc_gt_loc = 0

        # Combine the results
        for alm_file in alm_files:
            grad_t, mc_gt_sq = self._process_file_step(alm_loader, alm_file, **kwargs)

            mc_gt_loc += grad_t
            mc_gt_sq = utils.contract_almxblm(
                grad_t, self.icov(self.beam(np.conj(grad_t)))
            )
            mc_gt_sq_loc += mc_gt_sq
            mc_idx_loc += 1

        self.mc_gt = mc_gt_loc
        self.mc_gt_sq = mc_gt_sq_loc
        self.mc_idx = mc_idx_loc

    def compute_estimate_batch(self, alm_loader, alm_files, **kwargs):
        estimates = np.zeros(len(alm_files))
        fisher = kwargs.pop("fisher", self.compute_fisher())

        # Split alm_file loop over ranks.
        for aidx in range(len(alm_files)):
            alm_file = alm_files[aidx]
            alm = alm_loader(alm_file)

            estimate = self.compute_estimate(alm, fisher=fisher, **kwargs)
            logger.info("estimate: {}".format(estimate))

            estimates[aidx] = estimate

        return estimates

    def compute_estimate(self, alm, theta_batch=25, fisher=None, lin_term=None):
        # Similar to step, but only do backward transform, multiply alm with linear term
        # and apply normalization.

        alm = utils.alm_return_2d(alm, self.npol, self.lmax)
        alm = self.icov(alm)

        t_cubic = 0  # The cubic estimate.
        if fisher is None:
            fisher = self.compute_fisher()
        if lin_term is None:
            lin_term = self.compute_linear_term(alm, no_icov=True)

        a_ell_m = utils.alm2a_ell_m(alm)
        a_ell_m = a_ell_m.astype(self.cdtype)

        red_bisp = self.red_bispectra[0]
        f_i_ell, rule, weights = self._init_reduced_bispectrum(red_bisp)

        estimates = Parallel(n_jobs=-1, return_as="generator")(
            delayed(process_batch_estimate)(
                tidx_start,
                theta_batch,
                self.thetas,
                self.theta_weights,
                self.lmax,
                self.dtype,
                rule,
                weights,
                f_i_ell,
                a_ell_m,
                self.nphi,
            )
            for tidx_start in range(0, len(self.thetas), theta_batch)
        )

        t_cubic = sum(estimates)
        return (t_cubic - lin_term) / fisher

## utils

In [19]:
# def get_beamfunc(s):
# l, _ = hp.sphtfunc.Alm.getlm(s.lmax)
# sigma = s.beam_width / np.sqrt(8 * np.log(2))
# factor = np.exp(-(l**2) * sigma**2 / 2)

# def _beam(alm):
#     # alm -> alm exp(-l^2 sigma^2 / 2)
#     return hp.almxfl(alm, factor)

# return _beam


def get_beamfunc(s):
    if not s.noise:
        return lambda alm: alm
    else:
        beam_ell_pre = hp.gauss_beam(s.beam_width, lmax=s.lmax, pol=False)

        def _beam(alm):
            return np.array([hp.almxfl(alm[0], beam_ell_pre)])

        return _beam


def setup_ksw(s):
    camb_params_obj = camb.set_params(**s.cosmo_params)
    cosmo = Cosmology(camb_params_obj)
    cosmo.compute_transfer(s.cosmo_params["max_l"])
    cosmo.compute_c_ell()

    # create the local shape
    loc_shape = Shape.prim_local(s.cosmo_params["ns"], s.cosmo_params["pivot_scalar"])
    cosmo.add_prim_reduced_bispectrum(loc_shape, s.radii)

    # setup the data and get our icov object
    data = Data(s.lmax, s.noise_ell, s.beam_ell, s.pols, cosmo)
    icov = data.icov_diag_lensed if s.lensing else data.icov_diag_nonlensed
    ksw = KSW2(cosmo.red_bispectra, icov, get_beamfunc(s), s.lmax, s.pols)

    return ksw, data, icov, cosmo

## Heidelberg

In [20]:
s = Config(["settings/heidelberg.json", "--no-noise"])
ksw, data, icov, cosmo = setup_ksw(s)

10-Apr-24 14:29:35 - utils.config - INFO - Loading settings from file settings/heidelberg.json
10-Apr-24 14:29:35 - utils.config - INFO - Running with settings: 
{
  "cosmo_params": {
    "As": 2.457e-09,
    "ns": 0.96,
    "pivot_scalar": 0.05,
    "max_l": 1500,
    "lmax": 1024,
    "H0": 70.1,
    "ombh2": 0.02256,
    "omch2": 0.1143,
    "tau": 0.084
  },
  "nsims": 1000,
  "fnl_range": [
    -1000,
    1000
  ],
  "nside": 512,
  "noise_scale_tt": 500,
  "beam_width": 10,
  "lensing": false,
  "noise": false
}
10-Apr-24 14:29:35 - utils.config - INFO - Using seed 247534939
10-Apr-24 14:29:35 - utils.config - INFO - Running in SLURM job 11246776


In [21]:
# def compute_icov_ell(s, N, b, cosmo):
#     S_ell = cosmo._camb_data.get_cmb_power_spectra(
#         cosmo.camb_params,
#         lmax=s.lmax,
#         spectra=["total"],
#         CMB_unit="muK",
#         raw_cl=True,
#     )["total"][:, 0]
#     b_inv = 1 / b
#     return (1 / (S_ell + b_inv * N * b_inv))[None, :]


# def get_fisher_iso(s, ksw, cosmo):
#     icov_ell = compute_icov_ell(s, s.noise_ell, s.beam_ell, cosmo)
#     fisher_iso = ksw.compute_fisher_isotropic(icov_ell)
#     return fisher_iso, np.sqrt(1 / fisher_iso)

# get_fisher_iso(s, ksw, cosmo)

In [22]:
def alm_step_loader(idx):
    return data.compute_alm_sim(s.lensing)


mc_path = os.path.join(s.alm_dir, "kswmc")
os.makedirs(mc_path, exist_ok=True)
mc_file = os.path.join(mc_path, f"{s.base_name}_mc.hdf5")

if os.path.exists(mc_file):
    os.remove(mc_file)

if os.path.exists(mc_file):
    logger.info("Loading KSW state from %s", mc_file)
    ksw.start_from_read_state(mc_file, None)
else:
    logger.info("Running KSW step")
    thetas = int(np.floor(1.5 * s.lmax + 1))
    ksw.step_batch(alm_step_loader, range(100), theta_batch=thetas // 64)

    logger.info("Saving KSW state to %s", mc_file)
    ksw.write_state(mc_file)

10-Apr-24 14:30:14 - KSW2 - INFO - Running KSW step
10-Apr-24 14:30:14 - KSW2 - INFO - processing 0
10-Apr-24 14:30:36 - KSW2 - INFO - processing 1
10-Apr-24 14:30:52 - KSW2 - INFO - processing 2
10-Apr-24 14:31:10 - KSW2 - INFO - processing 3
10-Apr-24 14:31:26 - KSW2 - INFO - processing 4
10-Apr-24 14:31:42 - KSW2 - INFO - processing 5
10-Apr-24 14:31:59 - KSW2 - INFO - processing 6
10-Apr-24 14:32:15 - KSW2 - INFO - processing 7
10-Apr-24 14:32:31 - KSW2 - INFO - processing 8
10-Apr-24 14:32:47 - KSW2 - INFO - processing 9
10-Apr-24 14:33:04 - KSW2 - INFO - processing 10
10-Apr-24 14:33:20 - KSW2 - INFO - processing 11
10-Apr-24 14:33:34 - KSW2 - INFO - processing 12
10-Apr-24 14:33:50 - KSW2 - INFO - processing 13
10-Apr-24 14:34:07 - KSW2 - INFO - processing 14
10-Apr-24 14:34:24 - KSW2 - INFO - processing 15
10-Apr-24 14:34:40 - KSW2 - INFO - processing 16
10-Apr-24 14:34:56 - KSW2 - INFO - processing 17
10-Apr-24 14:35:12 - KSW2 - INFO - processing 18
10-Apr-24 14:35:28 - KSW2 -

In [ ]:
fisher = float(ksw.compute_fisher())
fisher, np.sqrt(1 / fisher)

In [ ]:
fnls = np.random.uniform(s.fnl_min, s.fnl_max + 1, s.total_sims)


def alm_hei_loader(idx):
    str_idx = str(idx).zfill(4)
    base1 = f"data/heidelberg/alm_l_{str_idx}_v3.fits"
    base2 = f"data/heidelberg/alm_nl_{str_idx}_v3.fits"

    alm_heidelberg_l = np.array(hp.read_alm(base1, hdu=1))
    alm_heidelberg_nl = np.array(hp.read_alm(base2, hdu=1))
    fnl = fnls[idx]

    alm = alm_heidelberg_l + fnl * alm_heidelberg_nl
    alm = alm * 2.7255 * 10 ** (6)  # convert heidelberg to uK
    alm = remove_mono_dipole(alm)

    logger.info("sending idx: %s, fnl: %s" % (idx, fnl))
    return alm


idxs = range(1, 51)
hei_estimates = ksw.compute_estimate_batch(alm_hei_loader, idxs, fisher=fisher)

In [ ]:
plot_dir = os.path.join(s.plot_dir, "notebooks")
os.makedirs(plot_dir, exist_ok=True)
pred_file = os.path.join(plot_dir, f"{s.base_name}_preds.png")
plot_ksw_predictions(
    fnls[idxs],
    hei_estimates,
    fisher,
    # save_file=pred_file,
)

## Sim

In [ ]:
s = Config(
    [
        "settings/heidelberg.json",
        "--nsims",
        "200",
        "--narray",
        "500",
        "--lensing",
        "--noise",
    ]
)

ksw, data, icov, cosmo = setup_ksw(s)

In [ ]:
# get_fisher_iso(s, ksw, cosmo)

In [ ]:
def alm_step_loader(idx):
    return data.compute_alm_sim(s.lensing)


mc_path = os.path.join(s.alm_dir, "kswmc")
os.makedirs(mc_path, exist_ok=True)
mc_file = os.path.join(mc_path, f"{s.base_name}_mc.hdf5")
if os.path.exists(mc_file):
    logger.info("Loading KSW state from %s", mc_file)
    ksw.start_from_read_state(mc_file, None)
else:
    logger.info("Running KSW step")
    thetas = int(np.floor(1.5 * s.lmax + 1))
    ksw.step_batch(alm_step_loader, range(100), theta_batch=thetas // 32)
    ksw_ran = True  # dont need to do this later

    logger.info("Saving KSW state to %s", mc_file)
    ksw.write_state(mc_file)

In [ ]:
fisher = float(ksw.compute_fisher())
fisher, np.sqrt(1 / fisher)

In [ ]:
alm_file = h5py.File(s.alm_file, "r", swmr=True, locking=False)
alms = alm_file["alm"]
fnls = alm_file["fnl"]


def alm_sim_loader(str_idx):
    """Loads in a single alm given a int in string form. Used inside the KSW code."""
    idx, pol = np.unravel_index(int(str_idx), (s.nsims, s.npol))
    alm = alms[idx, pol]
    return alm


idxs = range(50)
sim_estimates = ksw.compute_estimate_batch(alm_sim_loader, idxs, fisher=fisher)

In [ ]:
for idx, fnl, est in zip(idxs, fnls, hei_estimates):
    logger.info(f"idx: {idx}, fnl: {fnl}, est: {est}")
idx, pol = np.unravel_index(idxs, (s.nsims, s.npol))
plot_ksw_predictions(fnls[idx, 0], sim_estimates, fisher)